# 06 - Gold Layer - Date Dimension

Create a business-ready Date Dimension from the validated Silver calendar data.

**Source:** `end-to-end_pipeline.silver.calendar`
**Target:** `end-to-end_pipeline.gold.dim_date`

**Model Role:** Dimension Table
**Business Key:** `date`
**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide standardized date attributes for time-based analysis such as year, quarter, month, week, weekday, and weekend analysis.

---

## Cell 1 - Profile Silver Calendar Data

**Description:**
Confirm that the Silver calendar table is complete and suitable for the Gold Date Dimension. This checks date uniqueness, row count, range, and availability of the main time attributes.

```sql
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER CALENDAR FOR GOLD MODELING
-- Purpose: Confirm date grain, uniqueness,
--          completeness, and date range
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date) AS distinct_dates,
    COUNT(*) - COUNT(DISTINCT date) AS duplicate_dates,

    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)
        AS null_dates,

    COUNT(DISTINCT year)
        AS years,

    COUNT(DISTINCT quarter)
        AS quarters,

    COUNT(DISTINCT month_number)
        AS months,

    COUNT(DISTINCT day_name)
        AS day_names,

    MIN(date) AS earliest_date,
    MAX(date) AS latest_date

FROM `end-to-end_pipeline`.silver.calendar;
```

---

## Cell 2 - Inspect Date Business Attributes

**Description:**
Review the distribution of dates by year and quarter. This confirms that the Date Dimension supports historical and year-over-year analysis across the full 2021–2025 period.

```sql
%sql

-- ============================================================
-- CELL 2: INSPECT DATE BUSINESS ATTRIBUTES
-- Purpose: Review date distribution by year and quarter
-- ============================================================

SELECT
    year,
    quarter,
    COUNT(*) AS number_of_days

FROM `end-to-end_pipeline`.silver.calendar

GROUP BY
    year,
    quarter

ORDER BY
    year,
    quarter;
```

---

## Cell 3 - Transform Silver → Gold Date Dimension

**Description:**
Create the Gold Date Dimension at **one row per calendar date**.

Silver cleaning is not repeated. This step exposes the time attributes needed for business reporting and keeps `date` as the key that will later connect to `gold.fact_sales`.

```sql
%sql

-- ============================================================
-- CELL 3: CREATE GOLD DATE DIMENSION
-- Grain: One row per calendar date
-- Business Key: date
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_date AS

SELECT
    date,
    year,
    quarter,
    month_number,
    month_name,
    week_number,
    day_name,
    is_weekend

FROM `end-to-end_pipeline`.silver.calendar;
```

---

## Cell 4 - Validate Gold Date Dimension

**Description:**
Validate that the Gold Date Dimension contains one unique row per date, covers the complete 2021–2025 period, and contains no gaps in the calendar.

```sql
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD DATE DIMENSION
-- Purpose: Confirm date uniqueness, completeness,
--          continuity, and Silver → Gold consistency
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT date)
            AS distinct_dates,

        COUNT(*) - COUNT(DISTINCT date)
            AS duplicate_dates,

        SUM(
            CASE
                WHEN date IS NULL THEN 1
                ELSE 0
            END
        ) AS null_dates,

        COUNT(DISTINCT year)
            AS year_count,

        COUNT(DISTINCT quarter)
            AS quarter_count,

        COUNT(DISTINCT month_number)
            AS month_count,

        COUNT(DISTINCT day_name)
            AS day_name_count,

        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,

        DATEDIFF(MAX(date), MIN(date)) + 1 - COUNT(*)
            AS missing_dates

    FROM `end-to-end_pipeline`.gold.dim_date
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.calendar
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.total_rows = 1826
            AND v.distinct_dates = 1826
            AND v.duplicate_dates = 0
            AND v.null_dates = 0
            AND v.year_count = 5
            AND v.quarter_count = 4
            AND v.month_count = 12
            AND v.day_name_count = 7
            AND v.missing_dates = 0
            AND v.earliest_date = DATE '2021-01-01'
            AND v.latest_date = DATE '2025-12-31'
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;
```

If this returns **`PASS`**, then `06_gold_dim_date` is complete.

After this, the big one is **`01_gold_fact_sales`** — that’s where we’ll finally create the business measures like revenue, cost, discount amount, and profit, and connect the fact keys to your Gold dimensions.
